<a href="https://colab.research.google.com/github/mkane968/digital-text-methods/blob/main/Tutorial_4_Clean_and_Preprocess_Texts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 4: Cleaning and Preprocessing Text

In the previous tutorials, we learned how to represent textual data in Python and how to collect text from files, webpages, and digital archives.

Once we have collected our texts, they may not yet be ready for analysis. Digital texts often contain inconsistent capitalization, extra whitespace, punctuation, numbers, headers, or different forms of the same word.

**Text preprocessing** refers to the decisions and transformations we make to prepare textual data for analysis.

Importantly, preprocessing is not simply about making text "clean." Every preprocessing decision can preserve, change, or remove information. The appropriate choices depend on your **research question**.

### In this tutorial, we will practice:
- cleaning simple textual patterns with string methods and regular expressions
- normalizing capitalization and whitespace
- tokenizing text
- considering punctuation and numbers
- identifying and removing stopwords
- lemmatizing words with spaCy
- applying preprocessing to texts stored in a DataFrame
- preserving raw text alongside processed versions


## 1. Starting with Raw Text

Let's begin with a short example. Run the cell below and inspect the text before changing anything.


In [1]:
text = """
CHAPTER I

The children were running through the gardens.

A child stopped and shouted, "Look! There are 3 birds in the trees!"

The other children didn't stop running.
"""

print(text)



CHAPTER I

The children were running through the gardens.

A child stopped and shouted, "Look! There are 3 birds in the trees!"

The other children didn't stop running.



Before changing anything, consider what the text contains:

- blank lines and extra whitespace
- uppercase and lowercase letters
- punctuation and quotation marks
- a number
- different forms of related words (`child`, `children`)
- different forms of verbs (`running`, `stopped`)

Whether these features need to be changed depends on what we want to study.


## 2. Lowercasing

One common preprocessing step is converting text to lowercase. We practiced the `.lower()` method in Tutorial 1.


In [2]:
text_lower = text.lower()

print(text_lower)



chapter i

the children were running through the gardens.

a child stopped and shouted, "look! there are 3 birds in the trees!"

the other children didn't stop running.



Lowercasing allows `The` and `the` to be treated as the same string. This is often useful for frequency analysis.

However, capitalization may also contain useful information—for example, proper nouns or sentence beginnings.

**Before transforming a text, ask whether the information you are removing could matter to your research question.**


## 3. Cleaning with Regular Expressions

Sometimes we need to identify or clean **patterns** rather than exact words. For this, Python provides **regular expressions**, usually called **regex**.

Import Python's `re` package:


In [4]:
import re


Regular expressions describe patterns in text. For example, `\d` represents a digit.

We can use `re.findall()` to find patterns:


In [5]:
re.findall(r"\d+", text)


['3']

Here:

- `\d` means a digit
- `+` means one or more

So `\d+` means: **find one or more consecutive digits**.

A few useful patterns:

| Pattern | Meaning |
| --- | --- |
| `\d` | any digit |
| `\d+` | one or more digits |
| `\d{4}` | exactly four digits |
| `\s` | whitespace |
| `\s+` | one or more whitespace characters |
| `\w+` | one or more word characters |

You do not need to memorize regex syntax. The important idea is that regex allows us to identify **patterns within strings**.


## 4. Replacing Patterns with `re.sub()`

One common problem in digitized or web-extracted text is inconsistent whitespace.

We can replace repeated whitespace with a single space:


In [6]:
text_clean = re.sub(r"\s+", " ", text).strip()

print(text_clean)


CHAPTER I The children were running through the gardens. A child stopped and shouted, "Look! There are 3 birds in the trees!" The other children didn't stop running.


`re.sub()` means **substitute**. Its basic structure is:

`re.sub(pattern, replacement, text)`

Here, `\s+` identifies repeated whitespace and `" "` replaces it with a single space.

This is particularly useful with text extracted from HTML, XML, OCR, or other digital sources.


## 5. Removing a Repeated Pattern

Regex can also help when a corpus contains predictable material that we do not want to analyze.

For example, our sample begins with a chapter heading:


In [7]:
text_no_heading = re.sub(r"CHAPTER\s+[IVX]+", "", text)

print(text_no_heading)





The children were running through the gardens.

A child stopped and shouted, "Look! There are 3 birds in the trees!"

The other children didn't stop running.



This works because the heading follows a predictable pattern.

However, avoid removing material simply because it looks inconvenient. A chapter heading, page number, date, or other feature may be meaningful for some projects.


## 6. Tokenization

Many forms of text analysis require us to divide text into smaller units called **tokens**.

A simple way to create word-like tokens is `.split()`:


In [8]:
words = text_clean.lower().split()

words


['chapter',
 'i',
 'the',
 'children',
 'were',
 'running',
 'through',
 'the',
 'gardens.',
 'a',
 'child',
 'stopped',
 'and',
 'shouted,',
 '"look!',
 'there',
 'are',
 '3',
 'birds',
 'in',
 'the',
 'trees!"',
 'the',
 'other',
 'children',
 "didn't",
 'stop',
 'running.']

In [9]:
len(words)


28

`.split()` separates the string at whitespace. It is useful for seeing the basic idea of tokenization, but notice that punctuation may remain attached to words.

For example, Python may represent `"gardens."` and `"gardens"` as different strings.

For more sophisticated tokenization, we can use a natural language processing library.


## 7. Using spaCy

**spaCy** is a natural language processing library that can tokenize text and provide additional linguistic information.

Run the following cells:


In [10]:
import spacy

nlp = spacy.load("en_core_web_sm")


In [11]:
doc = nlp(text_clean)

for token in doc:
    print(token.text)


CHAPTER
I
The
children
were
running
through
the
gardens
.
A
child
stopped
and
shouted
,
"
Look
!
There
are
3
birds
in
the
trees
!
"
The
other
children
did
n't
stop
running
.


spaCy recognizes punctuation as separate tokens rather than leaving it attached to neighboring words.

Each token also contains additional information that we can use during preprocessing.


## 8. Punctuation and Numbers

spaCy can tell us whether a token is punctuation:


In [12]:
for token in doc:
    print(token.text, token.is_punct)


CHAPTER False
I False
The False
children False
were False
running False
through False
the False
gardens False
. True
A False
child False
stopped False
and False
shouted False
, True
" True
Look False
! True
There False
are False
3 False
birds False
in False
the False
trees False
! True
" True
The False
other False
children False
did False
n't False
stop False
running False
. True


We can use this information to keep only tokens that are not punctuation:


In [14]:
words_no_punct = [
    token.text.lower()
    for token in doc
    if not token.is_punct
]

words_no_punct


['chapter',
 'i',
 'the',
 'children',
 'were',
 'running',
 'through',
 'the',
 'gardens',
 'a',
 'child',
 'stopped',
 'and',
 'shouted',
 'look',
 'there',
 'are',
 '3',
 'birds',
 'in',
 'the',
 'trees',
 'the',
 'other',
 'children',
 'did',
 "n't",
 'stop',
 'running']

We can also identify alphabetic tokens using `token.is_alpha`:


In [15]:
words_alpha = [
    token.text.lower()
    for token in doc
    if token.is_alpha
]

words_alpha


['chapter',
 'i',
 'the',
 'children',
 'were',
 'running',
 'through',
 'the',
 'gardens',
 'a',
 'child',
 'stopped',
 'and',
 'shouted',
 'look',
 'there',
 'are',
 'birds',
 'in',
 'the',
 'trees',
 'the',
 'other',
 'children',
 'did',
 'stop',
 'running']

This removes punctuation and numbers.

Whether that is appropriate depends on the project. Numbers may be important if you are studying dates, prices, ages, quantities, or other numerical language.


## 9. Stopwords

Words such as `the`, `a`, `and`, `in`, and `to` occur frequently across English texts. These are often called **stopwords**.

spaCy identifies common stopwords for us:


In [16]:
for token in doc:
    print(token.text, token.is_stop)


CHAPTER False
I True
The True
children False
were True
running False
through True
the True
gardens False
. False
A True
child False
stopped False
and True
shouted False
, False
" False
Look False
! False
There True
are True
3 False
birds False
in True
the True
trees False
! False
" False
The True
other True
children False
did True
n't True
stop False
running False
. False


We can remove them:


In [17]:
words_no_stopwords = [
    token.text.lower()
    for token in doc
    if token.is_alpha and not token.is_stop
]

words_no_stopwords


['chapter',
 'children',
 'running',
 'gardens',
 'child',
 'stopped',
 'shouted',
 'look',
 'birds',
 'trees',
 'children',
 'stop',
 'running']

### A Caution About Stopwords

Stopwords are **not meaningless words**.

Words such as `I`, `we`, `you`, `they`, `not`, and `but` may reveal perspective, stance, audience, negation, or relationships among speakers.

Removing stopwords is therefore a **methodological choice**, not a universal cleaning requirement.


## 10. Lemmatization

Humans recognize relationships among forms such as:

- `run`, `runs`, `running`
- `child`, `children`

Python normally treats these as different strings.

**Lemmatization** reduces a word to a base or dictionary form called its **lemma**.

spaCy provides a lemma for each token:


In [18]:
for token in doc:
    print(token.text, "→", token.lemma_)


CHAPTER → CHAPTER
I → I
The → the
children → child
were → be
running → run
through → through
the → the
gardens → garden
. → .
A → a
child → child
stopped → stop
and → and
shouted → shout
, → ,
" → "
Look → look
! → !
There → there
are → be
3 → 3
birds → bird
in → in
the → the
trees → tree
! → !
" → "
The → the
other → other
children → child
did → do
n't → not
stop → stop
running → run
. → .


Look for words such as `children`, `running`, `was`, and `stopped`.

We can create a list of lemmas:


In [19]:
lemmas = [
    token.lemma_.lower()
    for token in doc
    if token.is_alpha
]

lemmas


['chapter',
 'i',
 'the',
 'child',
 'be',
 'run',
 'through',
 'the',
 'garden',
 'a',
 'child',
 'stop',
 'and',
 'shout',
 'look',
 'there',
 'be',
 'bird',
 'in',
 'the',
 'tree',
 'the',
 'other',
 'child',
 'do',
 'stop',
 'run']

Lemmatization can be useful when we want related grammatical forms to count together.

But word form itself may be important for some research questions, so lemmatization should also be a deliberate choice.


## 11. Combining Preprocessing Steps

We can combine several of the decisions above:


In [20]:
processed_words = [
    token.lemma_.lower()
    for token in doc
    if token.is_alpha
    and not token.is_stop
]

processed_words


['chapter',
 'child',
 'run',
 'garden',
 'child',
 'stop',
 'shout',
 'look',
 'bird',
 'tree',
 'child',
 'stop',
 'run']

This version:

1. tokenizes the text
2. keeps alphabetic tokens
3. removes stopwords
4. converts words to lemmas
5. converts the lemmas to lowercase

This may look like a very "clean" version of our text—but it also contains less information than the original.

The goal is **not to perform every possible preprocessing step**. The goal is to select the transformations appropriate for your analysis.


## 12. Comparing Word Frequencies

Recall `Counter` from Tutorial 1. Let's compare frequencies after preprocessing:


In [21]:
from collections import Counter

Counter(processed_words).most_common(10)


[('child', 3),
 ('run', 2),
 ('stop', 2),
 ('chapter', 1),
 ('garden', 1),
 ('shout', 1),
 ('look', 1),
 ('bird', 1),
 ('tree', 1)]

Compare these results with frequencies from a minimally processed version:


In [22]:
simple_words = text_clean.lower().split()

Counter(simple_words).most_common(10)


[('the', 4),
 ('children', 2),
 ('chapter', 1),
 ('i', 1),
 ('were', 1),
 ('running', 1),
 ('through', 1),
 ('gardens.', 1),
 ('a', 1),
 ('child', 1)]

Consider:

- Which words became more visible after preprocessing?
- Which information disappeared?
- Which version would be more useful for a frequency analysis?
- Would the answer change for a study of style, dialogue, pronouns, or punctuation?


## 13. Preprocessing Text in a DataFrame

A corpus will usually contain multiple texts rather than one string.

Let's create a small example DataFrame:


In [23]:
import pandas as pd

authors = ["Author A", "Author B", "Author C", "Author D"]
genres = ["fiction", "fiction", "poetry", "poetry"]

texts = [
    "The children were running through the garden.",
    "A child runs toward the DARK house!",
    "Children played beneath the trees.",
    "The child was playing quietly in the garden."
]

df = pd.DataFrame(
    list(zip(authors, genres, texts)),
    columns=["author", "genre", "text"]
)

df


,author,genre,text
0,Author A,fiction,The children were running through the garden.
1,Author B,fiction,A child runs toward the DARK house!
2,Author C,poetry,Children played beneath the trees.
3,Author D,poetry,The child was playing quietly in the garden.


### Preserve Your Raw Text

Whenever possible, **do not overwrite your original text**.

Instead, create new columns containing transformed versions:


In [24]:
df["text_lower"] = df["text"].str.lower()

df


,author,genre,text,text_lower
0,Author A,fiction,The children were running through the garden.,the children were running through the garden.
1,Author B,fiction,A child runs toward the DARK house!,a child runs toward the dark house!
2,Author C,poetry,Children played beneath the trees.,children played beneath the trees.
3,Author D,poetry,The child was playing quietly in the garden.,the child was playing quietly in the garden.


Now the DataFrame contains both:

- `text` — the original textual data
- `text_lower` — a transformed version

Keeping the raw data allows us to return to it later and make different preprocessing decisions.


## 14. Creating a Preprocessing Function

If we want to perform the same steps on every text, we can create a **function**.

A function is a reusable set of instructions.

You do not need to memorize the syntax yet. Focus on what goes into the function and what comes out.


In [25]:
def preprocess(text):
    doc = nlp(text)

    words = [
        token.lemma_.lower()
        for token in doc
        if token.is_alpha
        and not token.is_stop
    ]

    return words


Test it on one text:


In [26]:
preprocess("The children were running through the gardens!")


['child', 'run', 'garden']

Now apply the same function to every text in the DataFrame:


In [27]:
df["tokens"] = df["text"].apply(preprocess)

df


,author,genre,text,text_lower,tokens
0,Author A,fiction,The children were running through the garden.,the children were running through the garden.,"[child, run, garden]"
1,Author B,fiction,A child runs toward the DARK house!,a child runs toward the dark house!,"[child, run, dark, house]"
2,Author C,poetry,Children played beneath the trees.,children played beneath the trees.,"[child, play, beneath, tree]"
3,Author D,poetry,The child was playing quietly in the garden.,the child was playing quietly in the garden.,"[child, play, quietly, garden]"


Each row now preserves its original text while also containing a processed representation that we can use for analysis.


## 15. Counting Processed Tokens

Because the `tokens` column contains a list for each text, we can count the processed tokens:


In [28]:
df["token_count"] = df["tokens"].apply(len)

df


,author,genre,text,text_lower,tokens,token_count
0,Author A,fiction,The children were running through the garden.,the children were running through the garden.,"[child, run, garden]",3
1,Author B,fiction,A child runs toward the DARK house!,a child runs toward the dark house!,"[child, run, dark, house]",4
2,Author C,poetry,Children played beneath the trees.,children played beneath the trees.,"[child, play, beneath, tree]",4
3,Author D,poetry,The child was playing quietly in the garden.,the child was playing quietly in the garden.,"[child, play, quietly, garden]",4


We can also combine the tokens from all of our texts and examine corpus-level frequencies:


In [29]:
all_words = [
    word
    for tokens in df["tokens"]
    for word in tokens
]

Counter(all_words).most_common(10)


[('child', 4),
 ('run', 2),
 ('garden', 2),
 ('play', 2),
 ('dark', 1),
 ('house', 1),
 ('beneath', 1),
 ('tree', 1),
 ('quietly', 1)]

We have now moved through a basic preprocessing workflow:

**Raw Text → Cleaning → Tokenization → Linguistic Preprocessing → Processed Tokens → Analysis**

In later tutorials, we will use these representations to perform more systematic frequency and corpus analyses.


## 16. Preprocessing Is a Research Decision

Different research questions require different representations of a text.

| Research Interest | Information You Might Preserve |
| --- | --- |
| Vocabulary or topics | words and lemmas |
| Authorial style | punctuation, function words, capitalization |
| Character relationships | names and pronouns |
| Dialogue | quotation marks and sentence boundaries |
| Negation | *not*, *never*, *no* |
| Historical language | spelling and original word forms |
| Quantitative language | numbers and quantities |

Before preprocessing a corpus, ask:

1. **What textual feature am I trying to measure?**
2. **Which distinctions need to be preserved?**
3. **What information will each preprocessing step remove?**
4. **Can I justify that decision based on my research question?**


## Try It Yourself

Choose a short passage of approximately 100–200 words and store it as a string called `my_text`.

Then:

1. Inspect the raw text. What might need to be cleaned?
2. Use regex to normalize repeated whitespace.
3. Process the text with spaCy.
4. Create a list that removes punctuation but preserves stopwords.
5. Create a second list that removes stopwords.
6. Create a third list containing lemmas.
7. Calculate the ten most frequent words in at least two versions.

Then respond in a text cell:

**How did your preprocessing decisions change what appeared to be important or frequent in the text? Which preprocessing decisions would be appropriate for a research project using this text, and which would you avoid? Why?**


## From Preprocessing to Your Corpus

You will eventually make preprocessing decisions for your own corpus. There will not necessarily be one preprocessing pipeline that everyone in the class should use.

As you plan your project, consider:

1. **What do you actually need to identify or measure?**  
   Words, lemmas, phrases, sentences, names, punctuation, or something else?

2. **What cleaning does your source require?**  
   Does it contain headers, extra whitespace, page numbers, OCR artifacts, or other repeated material?

3. **Does capitalization matter?**

4. **Does punctuation matter?**

5. **Should you remove stopwords?**

6. **Do you want original word forms or lemmas?**

7. **What information must you preserve?**

Whenever possible, preserve your original texts and create separate processed versions.

The goal of preprocessing is **not to make a text as clean as possible**. It is to create a representation of the text that is appropriate for the question you want to investigate.


## References and Additional Resources

This tutorial draws on documentation and examples from the following resources:

- **spaCy 101**  
  https://spacy.io/usage/spacy-101  
  Introduction to spaCy's approach to tokenization, linguistic annotations, stopwords, and lemmas.

- **Python `re` Documentation**  
  https://docs.python.org/3/library/re.html  
  Official documentation for Python regular expressions.

- **pandas Documentation**  
  https://pandas.pydata.org/docs/  
  Official documentation for DataFrames and applying transformations to textual data.

- **NLTK Book: Natural Language Processing with Python**  
  https://www.nltk.org/book/  
  Steven Bird, Ewan Klein, and Edward Loper. An introduction to computational approaches to working with language.

- **Introduction to Cultural Analytics & Python**  
  https://melaniewalsh.github.io/Intro-Cultural-Analytics/  
  Melanie Walsh's open textbook introducing Python and computational methods for humanities and cultural research.

- **The Programming Historian — Python Tutorials**  
  https://programminghistorian.org/en/lessons/?topic=python  
  Peer-reviewed tutorials on Python and computational methods for humanities research.
